# Newton's Method as a Deep Computation
## Interactive Fractal Explorer

**Companion notebook** — *Complexity of Deep Computations via Topology* [Dueñez, Iovino, Matos-Wiederhold, Salvetti, Tall]

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eduenez/public-site/blob/main/2026-FES_Acatlan-ultracomputos/notebooks/newton_fractals.ipynb)

---

Newton's method is one of the oldest and most celebrated iterative algorithms.
Given a complex polynomial $p(z)$, the **Newton map** is
$$N_p(z) = z - \frac{p(z)}{p'(z)}.$$
Starting from any $z_0 \in \mathbb{C}$, we obtain the orbit
$$z_0 \;\xrightarrow{N_p}\; z_1 \;\xrightarrow{N_p}\; z_2 \;\xrightarrow{N_p}\; \cdots$$
This is a **deep computation**: a fixed transition rule applied indefinitely.
The central question is:

> *What does this orbit converge to — and how does the answer depend on $z_0$?*

The answer is extraordinarily rich, and it is the geometric prototype for the
**tame/wild dichotomy** studied in the accompanying research.


## Deep iterates and deep equilibria

After $n$ steps, $z_0$ has reached $z_n = N_p^{\circ n}(z_0)$ —
the **computation state at depth $n$**.

The fixed points of $N_p$ are exactly the **roots** $r_1, \ldots, r_d$ of $p$,
and they are the **deep equilibria**: once reached, the computation stays there.

For "most" starting points the orbit converges to some $r_k$.
But "most" conceals a remarkable subtlety.
Near the **Julia set** $\mathcal{J}(N_p)$, the deep-equilibrium map
$$\Phi : z_0 \;\longmapsto\; \lim_{n\to\infty} z_n$$
becomes *wildly discontinuous* — discontinuous on a dense set,
in every neighbourhood of every Julia-set point.

The interactive explorer below lets you see both regimes simultaneously.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import ipywidgets as widgets
%matplotlib inline


In [ ]:
# ── Polynomial presets ────────────────────────────────────────────────────────
#
# Each entry: (f, df, degree)
#   f, df : vectorised functions C -> C
#   degree: expected number of roots (for root-finder heuristic)

POLYNOMIALS = {
    "z^3 - 1   (symmetric, 3 roots)": (
        lambda z: z**3 - 1,
        lambda z: 3 * z**2,
        3,
    ),
    "z^3 - 2z + 2   (paper example)": (
        lambda z: z**3 - 2 * z + 2,
        lambda z: 3 * z**2 - 2,
        3,
    ),
    "z^4 - 1   (fourfold symmetry)": (
        lambda z: z**4 - 1,
        lambda z: 4 * z**3,
        4,
    ),
    "z^5 - 1   (fivefold symmetry)": (
        lambda z: z**5 - 1,
        lambda z: 5 * z**4,
        5,
    ),
    "z^5 - z + 0.376   (complex roots)": (
        lambda z: z**5 - z + 0.376,
        lambda z: 5 * z**4 - 1,
        5,
    ),
}


# ── Root finder ───────────────────────────────────────────────────────────────

def find_roots(f, df, degree, n_guesses=128):
    # Seed Newton's method from a circle; collect distinct roots.
    roots = []
    for k in range(n_guesses):
        z = 2.0 * np.exp(2j * np.pi * k / n_guesses)
        for _ in range(500):
            fz = f(z)
            if abs(fz) < 1e-13:
                break
            dfz = df(z)
            if abs(dfz) < 1e-15:
                break
            z -= fz / dfz
        if abs(f(z)) < 1e-8 and not any(abs(z - r) < 1e-5 for r in roots):
            roots.append(z)
        if len(roots) == degree:
            break
    return np.array(sorted(roots, key=np.angle))


# ── Core fractal computation ──────────────────────────────────────────────────

def compute_fractal(f, df, roots, cx, cy, width, res, max_iter, tol=1e-7):
    # Run Newton's method on a (res x res) complex grid centred at (cx, cy).
    # Returns:
    #   basin   (res, res) int8  -- root index; -1 if diverged / not converged
    #   n_iters (res, res) float -- iteration count (for smooth coloring)
    xs = np.linspace(cx - width / 2, cx + width / 2, res)
    ys = np.linspace(cy - width / 2, cy + width / 2, res)
    Z  = xs[np.newaxis, :] + 1j * ys[:, np.newaxis]

    basin   = np.full(Z.shape, -1, dtype=np.int8)
    n_iters = np.full(Z.shape, float(max_iter))
    active  = np.ones(Z.shape, dtype=bool)

    for n in range(max_iter):
        if not active.any():
            break
        with np.errstate(divide='ignore', invalid='ignore'):
            fz     = f(Z)
            dfz    = df(Z)
            safe_d = np.where(np.abs(dfz) > 1e-15, dfz, 1.0)
            step   = np.where(np.abs(dfz) > 1e-15, fz / safe_d, 0.0)
        Z[active] = (Z - step)[active]

        for k, r in enumerate(roots):
            newly   = active & (np.abs(Z - r) < tol)
            basin   = np.where(newly, k,       basin)
            n_iters = np.where(newly, n + 1.0, n_iters)
            active  = active & ~newly

        diverged = active & (np.abs(Z) > 1e6)
        n_iters  = np.where(diverged, n + 1.0, n_iters)
        active   = active & ~diverged

    return basin, n_iters


# ── Colorizer ────────────────────────────────────────────────────────────────
#
# Hue   = which root the pixel converged to
# Value = speed of convergence (bright=fast; dark=slow, near Julia set)
# Black = diverged or not converged within max_iter

_HUES = [0.00, 0.33, 0.63, 0.15, 0.50, 0.82]


def colorize(basin, n_iters, max_iter, mode):
    n_roots = int(basin.max()) + 1
    H = np.zeros(basin.shape)
    S = np.ones(basin.shape)
    V = np.zeros(basin.shape)   # black default

    for k in range(n_roots):
        mask = basin == k
        H[mask] = _HUES[k % len(_HUES)]
        if mode == "Smooth":
            V[mask] = 1.0 - (n_iters[mask] / max_iter) ** 0.45
        else:
            V[mask] = 0.85

    return mcolors.hsv_to_rgb(np.stack([H, S, V], axis=-1))


## Interactive explorer

Use the controls to explore Newton fractals for different polynomials,
iteration depths, and viewing windows.

**Suggested experiments**

1. **Iteration depth**: Start with *Max iterations = 5* and increase slowly.
   Watch the Julia-set boundary sharpen as the computation "resolves."
2. **Polynomial choice**: Compare $z^3 - 1$ (symmetric) with the paper example
   $z^3 - 2z + 2$ (asymmetric).
3. **Color mode**: Toggle *Smooth ↔ Basin*.
   In *Smooth* mode, brightness encodes convergence speed —
   dark pixels live near the Julia set, where convergence is slowest.
4. **Zoom**: Adjust *Center x/y* and *Width* to zoom into a boundary region.
   The fractal structure repeats at every scale.
5. **Higher degree**: Try $z^5 - 1$ for fivefold symmetry and richer boundaries.


In [ ]:
_root_cache = {}   # avoid recomputing roots on every slider move


@widgets.interact(
    polynomial=widgets.Dropdown(
        options=list(POLYNOMIALS.keys()),
        description='Polynomial',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='460px'),
    ),
    max_iter=widgets.IntSlider(
        value=60, min=5, max=300, step=5,
        description='Max iterations',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='460px'),
    ),
    resolution=widgets.Dropdown(
        options=[('Fast  (200x200)', 200),
                 ('Good  (400x400)', 400),
                 ('High  (600x600)', 600)],
        value=400,
        description='Resolution',
        style={'description_width': 'initial'},
    ),
    cx=widgets.FloatSlider(
        value=0.0, min=-3.0, max=3.0, step=0.05,
        description='Center x (Re)',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='460px'),
    ),
    cy=widgets.FloatSlider(
        value=0.0, min=-3.0, max=3.0, step=0.05,
        description='Center y (Im)',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='460px'),
    ),
    width=widgets.FloatSlider(
        value=4.0, min=0.05, max=8.0, step=0.05,
        description='Width (zoom)',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='460px'),
    ),
    color_mode=widgets.ToggleButtons(
        options=['Smooth', 'Basin'],
        description='Color mode',
        style={'description_width': 'initial'},
    ),
    show_roots=widgets.Checkbox(value=True, description='Mark roots'),
)
def explore(polynomial, max_iter, resolution, cx, cy, width, color_mode, show_roots):
    f, df, degree = POLYNOMIALS[polynomial]

    if polynomial not in _root_cache:
        _root_cache[polynomial] = find_roots(f, df, degree)
    roots = _root_cache[polynomial]

    basin, n_iters = compute_fractal(f, df, roots, cx, cy, width,
                                     resolution, max_iter)
    rgb = colorize(basin, n_iters, max_iter, color_mode)

    xmin, xmax = cx - width / 2, cx + width / 2
    ymin, ymax = cy - width / 2, cy + width / 2

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(rgb, extent=[xmin, xmax, ymin, ymax],
              origin='lower', interpolation='bilinear', aspect='equal')

    if show_roots:
        for k, r in enumerate(roots):
            if xmin < r.real < xmax and ymin < r.imag < ymax:
                ax.plot(r.real, r.imag, 'w*', markersize=14,
                        markeredgecolor='k', markeredgewidth=0.8, zorder=5)
                ax.annotate(
                    f'$r_{{{k+1}}}$',
                    xy=(r.real, r.imag),
                    xytext=(r.real + 0.06 * width, r.imag + 0.06 * width),
                    color='white', fontsize=12,
                    bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.55),
                    zorder=6,
                )

    short = polynomial.split('(')[0].strip()
    ax.set_xlabel('Re(z)', fontsize=12)
    ax.set_ylabel('Im(z)', fontsize=12)
    ax.set_title(
        f'Newton fractal: p(z) = {short}\n'
        f'({max_iter} iterations, {resolution}x{resolution} pixels)',
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()


## The Fatou–Julia decomposition

**Basins of attraction — the Fatou set.** For each root $r_k$ of $p$,
the set of starting points from which Newton's method converges to $r_k$ is
an open region $\mathcal{B}_k \subset \mathbb{C}$, called the *basin of attraction* of $r_k$.
Their union $\mathcal{F}(N_p) = \bigcup_k \mathcal{B}_k$ is the **Fatou set**.

In *Smooth* mode, brighter pixels converge faster;
darker pixels live near the Julia set where convergence slows.

**Julia set $\mathcal{J}(N_p)$.** The common boundary
$\mathcal{J} = \partial \mathcal{B}_k$ (independent of $k$)
is a closed, nowhere-dense, perfect set — a fractal.  On $\mathcal{J}$:
- Newton's method does **not** converge to any single root.
- The dynamics is **chaotic** and sensitive to initial conditions.
- Every open neighbourhood of a Julia-set point meets **all** basins.

**Historical note.** Cayley (1879) first studied Newton's method in the complex plane
and was puzzled by the intricate basin boundaries for degree $\geq 3$.
The full fractal theory was developed by Julia and Fatou in the 1910s–1920s,
long before "fractal" became a word.

> *Zoom into any boundary: you will find the same three-way intermingling
> of basins at every scale — a consequence of the chaotic dynamics on $\mathcal{J}$.*


## Connection to deep computations: tame vs. wild

### The deep-equilibrium map

Define the **deep-equilibrium map**
$$\Phi : \mathbb{C} \dashrightarrow \{r_1, \ldots, r_d\},
  \qquad \Phi(z_0) = \lim_{n \to \infty} N_p^{\circ n}(z_0).$$
This is well-defined on $\mathcal{F}(N_p)$ and undefined on $\mathcal{J}(N_p)$.

### On the Fatou set: $\Phi$ is Baire class 1 — *tame*

$\Phi|_{\mathcal{F}}$ is a **Baire class 1** function:
a pointwise limit of continuous functions (each $N_p^{\circ n}$ is continuous).
Equivalently, $\Phi|_{\mathcal{F}}$ satisfies the **NIP** (No Independence Property):
no infinite family of points in $\mathcal{F}$ can be "shattered" by the map $\Phi$.
NIP is the exact condition that guarantees **PAC-learnability** via uniform convergence.

### On the Julia set: $\Phi$ is not Baire class 1 — *wild*

Near $\mathcal{J}$, $\Phi$ fails to be Baire class 1.
In every neighbourhood of every Julia-set point, all $d$ roots appear densely —
an infinite independent pattern, the hallmark of the IP (Independence Property).

| Region | Baire class | NIP | Behaviour |
|--------|:-----------:|:---:|-----------|
| Basin interior $\mathcal{B}_k$ | Baire-1 | ✓ | Convergent, predictable |
| Julia set $\mathcal{J}(N_p)$ | Not Baire-1 | ✗ | Chaotic, sensitive |

### Why this matters for deep networks

A deep neural network is an iterated composition of layers.
As depth increases, the same questions arise:
*Does the computation converge?  Is the limit tractable?*

- **Tame (NIP) networks** have well-posed infinite-depth limits, approximable
  by polynomials and PAC-learnable.  This is the foundation of
  *Deep Equilibrium Models* (Bai, Kolter & Koltun, 2019).
- **Wild (IP) networks** exhibit Julia-set-like behaviour: chaotic sensitivity,
  no tractable limit, no reliable approximation.

The fractal boundary in the explorer above is a *visual model* for the divide
between tractable and intractable regimes in infinite-depth computation.

---
**Further exploration**

- How does the Julia set change as the polynomial degree increases?
- For $z^5 - z + c$, try different values of $c$ (edit the lambda above) and observe how the fractal varies.
- Can you find a polynomial whose Julia set looks qualitatively simpler?
  (Hint: for $z^2 - 1$ with only 2 roots, what happens?)
